In [1]:
import torch

# Check if CUDA is available
cuda_available = torch.cuda.is_available()

print(f"CUDA is available: {cuda_available}")

CUDA is available: True


In [2]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from transformers import BertModel, BertTokenizer
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import copy
from sklearn.metrics import classification_report, accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

c:\Users\k\.conda\envs\tf_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ==================== 配置区域 ====================

# 数据路径（请根据你的实际路径修改）
TRAIN_PATH = r"D:\②毕设\FRD-CLIP完整代码\tmp_input\train_ready.csv"
VAL_PATH = r"D:\②毕设\FRD-CLIP完整代码\tmp_input\val_ready.csv"
TEST_PATH = r"D:\②毕设\FRD-CLIP完整代码\tmp_input\test_ready.csv"

# BERT模型路径（使用中文BERT，请根据你的环境修改）
# 如果本地没有，可以改为 "bert-base-chinese" 自动下载
BERT_PATH = "bert-base-chinese"  # 或本地路径如：r"D:\models\bert-base-chinese"

# 训练配置
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 128          # 文本最大长度
BATCH_SIZE = 32        # 批次大小
EPOCHS = 5             # 训练轮数
LR = 2e-5              # 学习率
SEED = 42              # 随机种子

# 保存路径
SAVE_PATH = "best_bert_text_only.pth"

# 设置随机种子保证可复现
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"使用设备: {DEVICE}")

使用设备: cuda


In [4]:
# ==================== 1. 模型定义 ====================

class BERT_Text_Classifier(nn.Module):
    """
    纯BERT文本分类模型
    使用BERT提取文本特征，然后通过全连接层分类
    """
    def __init__(self, bert_path, num_classes=2, dropout=0.3):
        super(BERT_Text_Classifier, self).__init__()
        
        # 加载预训练BERT
        self.bert = BertModel.from_pretrained(bert_path)
        self.hidden_size = self.bert.config.hidden_size  # 通常是768
        
        # 分类器：Dropout + 全连接层
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, input_ids, attention_mask):
        """
        前向传播
        Args:
            input_ids: [batch_size, seq_len]
            attention_mask: [batch_size, seq_len]
        Returns:
            logits: [batch_size, num_classes]
        """
        # 获取BERT输出
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # 使用[CLS] token的输出作为句子表示 [batch_size, hidden_size]
        pooled_output = outputs.pooler_output
        
        # 分类
        logits = self.classifier(pooled_output)
        
        return logits

In [5]:
# ==================== 2. 数据集定义 ====================

class TextDataset(Dataset):
    """
    纯文本数据集
    只使用text_clean和label字段
    """
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        
        # 数据清洗：确保text_clean不为空
        self.texts = []
        self.labels = []
        
        for idx, row in self.df.iterrows():
            text = str(row["text_clean"]) if pd.notna(row["text_clean"]) else ""
            label = int(row["label"]) if pd.notna(row["label"]) else 0
            
            self.texts.append(text)
            self.labels.append(label)
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        # BERT tokenize
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,      # 添加[CLS]和[SEP]
            max_length=self.max_len,
            padding='max_length',         # 填充到max_length
            truncation=True,              # 超长截断
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),      # [max_len]
            'attention_mask': encoding['attention_mask'].squeeze(0),  # [max_len]
            'label': torch.tensor(label, dtype=torch.long)
        }

In [6]:
# ==================== 3. 评估函数 ====================

@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    """
    评估模型性能
    返回: (平均损失, 准确率, 所有标签, 所有预测)
    """
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_preds = []
    
    for batch in tqdm(dataloader, desc="Evaluating", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        
        # 前向传播
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        
        # 预测
        preds = torch.argmax(outputs, dim=1)
        
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return avg_loss, accuracy, f1, all_labels, all_preds


def print_classification_report(labels, preds, target_names=['real', 'fake']):
    """
    打印详细的分类报告
    """
    print("\n" + "="*60)
    print("分类报告:")
    print("-"*60)
    print(classification_report(labels, preds, target_names=target_names, digits=4))
    print(f"准确率 (Accuracy): {accuracy_score(labels, preds):.4f}")
    print(f"F1分数 (Weighted): {f1_score(labels, preds, average='weighted'):.4f}")
    print("="*60)

In [7]:
# ==================== 4. 训练函数 ====================

def train_epoch(model, dataloader, optimizer, criterion, device):
    """
    训练一个epoch
    """
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    progress_bar = tqdm(dataloader, desc="Training", leave=False)
    
    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        
        # 清零梯度
        optimizer.zero_grad()
        
        # 前向传播
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        
        # 反向传播
        loss.backward()
        
        # 梯度裁剪（防止梯度爆炸）
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # 更新参数
        optimizer.step()
        
        total_loss += loss.item()
        
        # 记录预测
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        # 更新进度条
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_preds)
    
    return avg_loss, accuracy


def train_model():
    """
    主训练函数
    """
    print("\n" + "="*60)
    print("开始加载数据和模型...")
    print("="*60)
    
    # 1. 加载tokenizer
    print(f"加载BERT Tokenizer: {BERT_PATH}")
    tokenizer = BertTokenizer.from_pretrained(BERT_PATH)
    
    # 2. 加载数据
    print("加载数据集...")
    train_df = pd.read_csv(TRAIN_PATH)
    val_df = pd.read_csv(VAL_PATH)
    test_df = pd.read_csv(TEST_PATH)
    
    print(f"训练集大小: {len(train_df)}")
    print(f"验证集大小: {len(val_df)}")
    print(f"测试集大小: {len(test_df)}")
    
    # 3. 创建数据集和DataLoader
    train_dataset = TextDataset(train_df, tokenizer, max_len=MAX_LEN)
    val_dataset = TextDataset(val_df, tokenizer, max_len=MAX_LEN)
    test_dataset = TextDataset(test_df, tokenizer, max_len=MAX_LEN)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True, 
        num_workers=0,  # Windows建议设为0
        pin_memory=True if torch.cuda.is_available() else False
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False,
        num_workers=0
    )
    test_loader = DataLoader(
        test_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False,
        num_workers=0
    )
    
    # 4. 初始化模型
    print(f"\n初始化BERT文本分类模型...")
    model = BERT_Text_Classifier(BERT_PATH, num_classes=2, dropout=0.3)
    model = model.to(DEVICE)
    
    # 统计参数量
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"总参数量: {total_params:,}")
    print(f"可训练参数量: {trainable_params:,}")
    
    # 5. 优化器和损失函数
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()
    
    # 学习率调度器（可选）
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=1, verbose=True
    )
    
    # 6. 训练循环
    print("\n" + "="*60)
    print("开始训练...")
    print("="*60)
    
    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [], 'val_f1': []
    }
    
    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch + 1}/{EPOCHS}")
        print("-" * 40)
        
        # 训练阶段
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
        
        # 验证阶段
        val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion, DEVICE)
        
        # 记录历史
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        
        # 打印结果
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} | Val F1: {val_f1:.4f}")
        
        # 学习率调整
        scheduler.step(val_acc)
        
        # 保存最优模型
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save({
                'epoch': epoch,
                'model_state_dict': best_model_wts,
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': best_val_acc,
            }, SAVE_PATH)
            print(f"*** 保存最优模型 (Val Acc: {best_val_acc:.4f}) ***")
    
    # 7. 加载最优模型并进行最终测试
    print("\n" + "="*60)
    print("训练完成！加载最优模型进行测试...")
    print("="*60)
    
    checkpoint = torch.load(SAVE_PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # 在验证集上最终评估
    print("\n【验证集结果】")
    val_loss, val_acc, val_f1, val_labels, val_preds = evaluate(model, val_loader, criterion, DEVICE)
    print_classification_report(val_labels, val_preds)
    
    # 在测试集上最终评估
    print("\n【测试集结果】")
    test_loss, test_acc, test_f1, test_labels, test_preds = evaluate(model, test_loader, criterion, DEVICE)
    print_classification_report(test_labels, test_preds)
    
    # 打印训练历史
    print("\n【训练历史】")
    for i in range(EPOCHS):
        print(f"Epoch {i+1}: "
              f"Train Loss={history['train_loss'][i]:.4f}, "
              f"Train Acc={history['train_acc'][i]:.4f}, "
              f"Val Acc={history['val_acc'][i]:.4f}")
    
    return model, history

In [8]:
# ==================== 主程序入口 ====================

if __name__ == "__main__":
    # 开始训练
    model, history = train_model()
    
    print("\n" + "="*60)
    print("所有任务已完成！")
    print(f"最优模型已保存至: {SAVE_PATH}")
    print("="*60)
    
    # 示例：使用训练好的模型进行单条预测
    # tokenizer = BertTokenizer.from_pretrained(BERT_PATH)
    # result = predict_single_text(model, tokenizer, "这是一条测试文本")
    # print(f"预测结果: {result}")


开始加载数据和模型...
加载BERT Tokenizer: bert-base-chinese
加载数据集...
训练集大小: 5453
验证集大小: 1168
测试集大小: 1169

初始化BERT文本分类模型...
总参数量: 102,465,026
可训练参数量: 102,465,026

开始训练...

Epoch 1/5
----------------------------------------


Train Loss: 0.3507 | Train Acc: 0.8617
Val Loss:   0.2412 | Val Acc:   0.9015 | Val F1: 0.9016
*** 保存最优模型 (Val Acc: 0.9015) ***

Epoch 2/5
----------------------------------------


Train Loss: 0.2347 | Train Acc: 0.9166
Val Loss:   0.2213 | Val Acc:   0.9144 | Val F1: 0.9139
*** 保存最优模型 (Val Acc: 0.9144) ***

Epoch 3/5
----------------------------------------


Train Loss: 0.1664 | Train Acc: 0.9415
Val Loss:   0.2336 | Val Acc:   0.9135 | Val F1: 0.9135

Epoch 4/5
----------------------------------------


Train Loss: 0.1221 | Train Acc: 0.9597
Val Loss:   0.2803 | Val Acc:   0.9127 | Val F1: 0.9126

Epoch 5/5
----------------------------------------


Train Loss: 0.0751 | Train Acc: 0.9749
Val Loss:   0.3341 | Val Acc:   0.9161 | Val F1: 0.9161
*** 保存最优模型 (Val Acc: 0.9161) ***

训练完成！加载最优模型进行测试...

【验证集结果】



分类报告:
------------------------------------------------------------
              precision    recall  f1-score   support

        real     0.9220    0.9220    0.9220       628
        fake     0.9093    0.9093    0.9093       540

    accuracy                         0.9161      1168
   macro avg     0.9156    0.9156    0.9156      1168
weighted avg     0.9161    0.9161    0.9161      1168

准确率 (Accuracy): 0.9161
F1分数 (Weighted): 0.9161

【测试集结果】



分类报告:
------------------------------------------------------------
              precision    recall  f1-score   support

        real     0.8994    0.9380    0.9183       629
        fake     0.9240    0.8778    0.9003       540

    accuracy                         0.9102      1169
   macro avg     0.9117    0.9079    0.9093      1169
weighted avg     0.9107    0.9102    0.9100      1169

准确率 (Accuracy): 0.9102
F1分数 (Weighted): 0.9100

【训练历史】
Epoch 1: Train Loss=0.3507, Train Acc=0.8617, Val Acc=0.9015
Epoch 2: Train Loss=0.2347, Train Acc=0.9166, Val Acc=0.9144
Epoch 3: Train Loss=0.1664, Train Acc=0.9415, Val Acc=0.9135
Epoch 4: Train Loss=0.1221, Train Acc=0.9597, Val Acc=0.9127
Epoch 5: Train Loss=0.0751, Train Acc=0.9749, Val Acc=0.9161

所有任务已完成！
最优模型已保存至: best_bert_text_only.pth
